In [ ]:
# ── Cell 1 : Imports ───────────────────────────────────────────────────────────
import json
import time
import datetime
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from matplotlib.lines import Line2D

import liesel.model as lsl
import liesel.goose as gs
from tensorflow_probability.substrates.jax.experimental import distributions as tfde
from tensorflow_probability.python.internal.backend.jax.compat import v2 as tf
import tensorflow_probability.substrates.jax.distributions as tfd
import tensorflow_probability.substrates.jax.bijectors as tfb

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
# ── Cell 2 : Data Loading ──────────────────────────────────────────────────────
data_path = "margarine_data.json"
print(f"Loading data from {data_path}...")

with open(data_path, "r") as f:
    lgtdata_list = json.load(f)

n_params      = 10
n_alts        = 10
K_components  = 3

X_list, y_list, unit_idx_list, Z_list = [], [], [], []

for i, hh_data in enumerate(lgtdata_list):
    y_i      = np.atleast_1d(hh_data["y"]) - 1          # 0-indexed
    X_i_flat = np.array(hh_data["X"])
    Z_raw    = hh_data["Z_raw"]

    n_obs = len(y_i)
    X_i   = X_i_flat.reshape((n_obs, n_alts, n_params))  # (n_obs, n_alts, n_params)

    for t in range(n_obs):
        X_list.append(X_i[t])
        y_list.append(y_i[t])
        unit_idx_list.append(i)

    Z_list.append(Z_raw)

# ── Demographics: log(Income), mean-centre, add intercept ─────────────────────
Z = np.array(Z_list, dtype=float)

valid_inc          = Z[:, 0] > 0
Z[valid_inc,  0]   = np.log(Z[valid_inc, 0])
Z[~valid_inc, 0]   = np.nan

Z_mean = np.nanmean(Z, axis=0)
Z      = Z - Z_mean
Z      = np.nan_to_num(Z)                                # NaN → 0 (= mean after centering)

Z_with_int = np.column_stack((np.ones(len(Z)), Z))       # intercept + log_income + fam_size

choice_data = {
    "X":        jnp.array(X_list),
    "y":        jnp.array(y_list),
    "Z":        jnp.array(Z_with_int),
    "unit_idx": jnp.array(unit_idx_list),
    "n_units":  len(lgtdata_list),
    "n_params": n_params,
    "n_demos":  Z_with_int.shape[1],   # 3: intercept + log_income + fam_size
    "K":        K_components,
    "n_alts":   n_alts,
}

N = choice_data["n_units"]
K = choice_data["K"]
P = choice_data["n_params"]
D = choice_data["n_demos"]
A = choice_data["n_alts"]

param_names = [f"ASC_{a+1}" for a in range(P - 1)] + ["price"]
demo_names  = ["intercept", "log_income", "fam_size"]

print(f"Households  : {N}")
print(f"Total obs   : {len(y_list)}")
print(f"Parameters  : {P}  (9 ASCs + price)")
print(f"Alternatives: {A}")
print(f"Demographics: {D}  (intercept, log_income, fam_size)")
print(f"Components  : {K}")
print(f"X shape     : {choice_data['X'].shape}")
print(f"Z shape     : {choice_data['Z'].shape}")

Loading data from margarine_data.json...
Households  : 516
Total obs   : 4470
Parameters  : 10  (9 ASCs + price)
Alternatives: 10
Demographics: 3  (intercept, log_income, fam_size)
Components  : 2
X shape     : (4470, 10, 10)
Z shape     : (516, 3)


In [ ]:
# ── Cell 3 : Model Definition ──────────────────────────────────────────────────

def make_wishart(df, scale_tril):
    return tfd.WishartTriL(
        df=df,
        scale_tril=scale_tril,
        input_output_cholesky=True,
        validate_args=False
    )


def make_mvn_precision(loc, precision_factor):
    return tfde.MultivariateNormalPrecisionFactorLinearOperator(
        loc=loc,
        precision_factor=tf.linalg.LinearOperatorLowerTriangular(precision_factor),
        validate_args=False
    )


def build_mixture_hmnl_model(
        data_dict,
        A_delta=0.01,
        a_mu=0.01,
        dirichlet_a=5.0
):
    """
    HMNL with mixture-of-normals heterogeneity.
    Prior structure matches bayesm::rhierMnlRwMixture.

        beta_i = Z[i] @ Delta + u_i,   u_i ~ N(mu_k, Sigma_k),  k ~ Categorical(pvec)

    Wishart:   Sigma_k^{-1} ~ W(nu, V^{-1}),  nu = n_params + 3,  V = nu * I
    Normal:    mu_k | Sigma_k ~ N(0, Sigma_k / a_mu)
    Normal:    Delta ~ N(0, (1/A_delta) * I)
    Dirichlet: pvec ~ Dir(dirichlet_a)
    """
    n_params = int(data_dict["n_params"])
    n_units  = int(data_dict["n_units"])
    K_comp   = int(data_dict["K"])
    has_Z    = data_dict.get("Z") is not None

    # ── Wishart prior ─────────────────────────────────────────────────────────
    nu          = float(n_params + 3)
    V           = nu * jnp.eye(n_params)
    Vinv_chol   = jnp.linalg.cholesky(jnp.linalg.inv(V))
    Vinv_chol_K = jnp.broadcast_to(Vinv_chol[None], (K_comp, n_params, n_params))

    # ── pvec ~ Dirichlet ──────────────────────────────────────────────────────
    pvec = lsl.Var.new_param(
        value=jnp.ones(K_comp) / K_comp,
        distribution=lsl.Dist(tfd.Dirichlet,
                               concentration=jnp.ones(K_comp) * dirichlet_a),
        name="pvec"
    )
    pvec_latent = pvec.transform(tfb.SoftmaxCentered(), name="pvec_latent")

    # ── Sigma_k^{-1} ~ Wishart via Cholesky ──────────────────────────────────
    sigma_inv_chol_k = lsl.Var.new_param(
        value=jnp.broadcast_to(jnp.eye(n_params)[None], (K_comp, n_params, n_params)),
        distribution=lsl.Dist(
            make_wishart,
            df=jnp.full(K_comp, nu),
            scale_tril=Vinv_chol_K
        ),
        name="sigma_inv_chol_k"
    )
    sigma_inv_chol_k_latent = sigma_inv_chol_k.transform(
        tfb.FillScaleTriL(), name="sigma_inv_chol_k_latent"
    )

    # ── mu_k | Sigma_k ~ N(0, Sigma_k / a_mu) ────────────────────────────────
    mu_prec_factor_k = lsl.Var.new_calc(
        lambda L: jnp.sqrt(a_mu) * L,
        L=sigma_inv_chol_k,
        name="mu_prec_factor_k"
    )
    mu_k = lsl.Var.new_param(
        value=jnp.zeros((K_comp, n_params)),
        distribution=lsl.Dist(
            make_mvn_precision,
            loc=jnp.zeros(n_params),
            precision_factor=mu_prec_factor_k
        ),
        name="mu_k"
    )

    # ── Delta ~ N(0, (1/A_delta) * I) ────────────────────────────────────────
    if has_Z:
        n_demos           = int(data_dict["Z"].shape[1])
        Z_var             = lsl.Var.new_obs(data_dict["Z"], name="Z_obs")
        Delta_prec_factor = jnp.sqrt(A_delta) * jnp.eye(n_params)

        Delta = lsl.Var.new_param(
            value=jnp.zeros((n_demos, n_params)),
            distribution=lsl.Dist(
                make_mvn_precision,
                loc=jnp.zeros(n_params),
                precision_factor=Delta_prec_factor
            ),
            name="Delta"
        )
        z_delta = lsl.Var.new_calc(
            lambda z, d: z @ d, z=Z_var, d=Delta, name="z_delta"
        )

    # ── Sigma_k (covariance) from precision Cholesky ──────────────────────────
    # sigma_chol_k = lsl.Var.new_calc(
    #     lambda L: jax.vmap(
    #         lambda Lk: jnp.linalg.cholesky(
    #             jnp.linalg.inv(Lk @ Lk.T) + 1e-6 * jnp.eye(n_params)
    #         )
    #     )(L),
    #     L=sigma_inv_chol_k,
    #     name="sigma_chol_k"
    # )

    # ── beta_i location: Z[i] @ Delta + mu_k  (n_units, K, n_params) ─────────
    if has_Z:
        beta_loc = lsl.Var.new_calc(
            lambda zd, mu: zd[:, None, :] + mu[None, :, :],
            zd=z_delta, mu=mu_k,
            name="beta_loc"
        )
    else:
        beta_loc = lsl.Var.new_calc(
            lambda mu: jnp.broadcast_to(mu[None, :, :], (n_units, K_comp, n_params)),
            mu=mu_k,
            name="beta_loc"
        )

    # Updated mixture function using precision factors
    def make_beta_mixture(pvec, locs, precision_factors):
        return tfd.MixtureSameFamily(
            mixture_distribution=tfd.Categorical(probs=pvec),
            components_distribution=tfde.MultivariateNormalPrecisionFactorLinearOperator(
                loc=locs,
                # We add [None] to broadcast the K components across the N units
                precision_factor=tf.linalg.LinearOperatorLowerTriangular(precision_factors[None])
            )
        )

    # Updated beta_i definition
    beta_i = lsl.Var.new_param(
        value=jnp.zeros((n_units, n_params)),
        distribution=lsl.Dist(
            make_beta_mixture,
            pvec=pvec,
            locs=beta_loc,
            precision_factors=sigma_inv_chol_k  # Pass the Cholesky factor of the precision directly
        ),
        name="beta_i"
    )

    # ── Likelihood ────────────────────────────────────────────────────────────
    X_var         = lsl.Var.new_obs(data_dict["X"],        name="X_obs")
    idx_var       = lsl.Var.new_obs(data_dict["unit_idx"], name="idx_obs")
    beta_expanded = lsl.Var.new_calc(
        lambda b, idx: b[idx], b=beta_i, idx=idx_var, name="beta_expanded"
    )
    logits = lsl.Var.new_calc(
        lambda x, b: jnp.einsum("nij,nj->ni", x, b),
        x=X_var, b=beta_expanded,
        name="logits"
    )
    y_var = lsl.Var.new_obs(
        data_dict["y"],
        distribution=lsl.Dist(tfd.Categorical, logits=logits),
        name="y"
    )

    return lsl.Model([y_var])


print("Building 2-component mixture HMNL model...")
hmnl_model = build_mixture_hmnl_model(
    choice_data, A_delta=0.01, a_mu=0.01, dirichlet_a=5.0
)
print("Model built successfully.")

Building 2-component mixture HMNL model...


c:\Users\ThinkPad\Desktop\Repositories\BDCM\liesel_project\.venv\Lib\site-packages\jax\_src\numpy\array_methods.py:122: UserWarning: Explicitly requested dtype float64 requested in astype is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)


Model built successfully.


In [ ]:
def run_mixture_inference(model, data_dict, chains=4, warmup=1000,
                          posterior=5000, seed=123):
    has_Z = data_dict.get("Z") is not None
    eb = gs.EngineBuilder(seed=seed, num_chains=chains)
    eb.set_model(gs.LieselInterface(model))
    eb.set_initial_values(model.state)

    # REVERTED: Back to separate kernels
    # Block 1: Hyperparameters (Fast, data-independent evaluations)
    eb.add_kernel(gs.NUTSKernel(
        ["pvec_latent", "mu_k", "sigma_inv_chol_k_latent"],
        mm_diag=True #change here tofor speedup, NUTS takes forever to return erroneous kernels 
    ))
    
    # Block 2: Covariates (if present)
    if has_Z:
        eb.add_kernel(gs.NUTSKernel(["Delta"]))

    # Block 3: Individual level parameters (Heavy data likelihood evaluated here)
    eb.add_kernel(gs.NUTSKernel(["beta_i"], mm_diag=True))

    eb.set_duration(warmup_duration=warmup, posterior_duration=posterior)

    print("Starting NUTS Sampling — 2-Component Mixture HMNL...")
    engine = eb.build()
    engine.sample_all_epochs()
    return engine.get_results(), engine.get_results().get_posterior_samples()

In [ ]:
start_time = time.time()
mcmc_results, posterior_samples = run_mixture_inference(
    hmnl_model, choice_data, chains=4, warmup=1000, posterior=2500, seed=123
)
print(f"Sampling finished in "
      f"{datetime.timedelta(seconds=int(time.time() - start_time))}")

liesel.goose.builder - WARNING - No jitter functions provided. The initial values won't be jittered
liesel.goose.engine - INFO - Initializing kernels...


Starting NUTS Sampling — 2-Component Mixture HMNL...


liesel.goose.engine - INFO - Done
liesel.goose.engine - INFO - Starting epoch: FAST_ADAPTATION, 75 transitions, 25 jitted together
100%|██████████████████████████████████████████| 3/3 [00:26<00:00,  8.93s/chunk]
liesel.goose.engine - WARNING - Errors per chain for kernel_00: 7, 5, 8, 3 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_01: 2, 2, 3, 4 / 75 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_02: 2, 2, 3, 2 / 75 transitions
liesel.goose.engine - INFO - Finished epoch
liesel.goose.engine - INFO - Starting epoch: SLOW_ADAPTATION, 25 transitions, 25 jitted together
100%|█████████████████████████████████████████| 1/1 [00:00<00:00, 308.25chunk/s]
liesel.goose.engine - WARNING - Errors per chain for kernel_00: 2, 3, 3, 1 / 25 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_01: 2, 1, 2, 1 / 25 transitions
liesel.goose.engine - WARNING - Errors per chain for kernel_02: 2, 1, 1, 1 / 25 transitions
liesel.goose.e

In [ ]:
# ── Cell 5 : Label Switching — Pivotal Reordering Algorithm ───────────────────

def pra(xi_star, xi_draws):
    m, K_c, J = xi_draws.shape
    perms     = np.empty((m, K_c), dtype=int)
    relabeled = np.empty_like(xi_draws)
    for t in range(m):
        cost = np.sum(
            (xi_star[:, None, :] - xi_draws[t, None, :, :]) ** 2, axis=-1
        )
        _, perms[t] = linear_sum_assignment(cost)
        relabeled[t] = xi_draws[t, perms[t], :]
    return perms, relabeled


def apply_permutations(draws, perms):
    relabeled = np.empty_like(draws)
    for t in range(perms.shape[0]):
        relabeled[t] = draws[t, perms[t]]
    return relabeled


def resolve_label_switching(samples_dict, pivot_chain_idx=0):
    print("\nStarting Pivotal Reordering Algorithm (PRA)...")
    mu                           = samples_dict["mu_k"]
    n_chains, n_draws, K_c, n_p = mu.shape
    m                            = n_chains * n_draws

    mu_flat  = mu.reshape(m, K_c, n_p)
    xi_star  = np.mean(mu[pivot_chain_idx], axis=0)
    perms, relabeled_mu_flat = pra(xi_star, mu_flat)

    sorted_samples         = samples_dict.copy()
    sorted_samples["mu_k"] = relabeled_mu_flat.reshape(n_chains, n_draws, K_c, n_p)

    sigma      = samples_dict["sigma_inv_chol_k_latent"]
    s_shape    = sigma.shape
    sigma_flat = sigma.reshape(m, K_c, *s_shape[3:])
    sorted_samples["sigma_inv_chol_k_latent"] = apply_permutations(
        sigma_flat, perms
    ).reshape(s_shape)

    pvec_simplex = np.array(
        tfb.SoftmaxCentered().forward(samples_dict["pvec_latent"])
    )
    pvec_flat = pvec_simplex.reshape(m, K_c)
    sorted_samples["pvec"] = apply_permutations(
        pvec_flat, perms
    ).reshape(n_chains, n_draws, K_c)

    print("Label switching resolved — all chains aligned.")
    return sorted_samples


posterior_samples_sorted = resolve_label_switching(
    posterior_samples, pivot_chain_idx=0
)

In [ ]:
# ── Cell 6 : Diagnostics ──────────────────────────────────────────────────────
print("\n=== MCMC Convergence Summary ===")
summary = gs.Summary(mcmc_results)


def plot_cholesky_traces(samples_dict, n_params, k_idx=0,
                         param_name="sigma_inv_chol_k_latent", figsize=(15, 12)):
    latent_samples          = samples_dict[param_name][:, :, k_idx, :]
    n_chains, n_draws, n_latent = latent_samples.shape

    fig, axes = plt.subplots(n_params, n_params, figsize=figsize, sharex=True)
    if n_params == 1:
        axes = np.array([[axes]])

    latent_idx = 0
    for i in range(n_params):
        for j in range(n_params):
            ax = axes[i, j]
            if i >= j and latent_idx < n_latent:
                for chain in range(n_chains):
                    ax.plot(latent_samples[chain, :, latent_idx],
                            lw=0.6, label=f"Chain {chain}")
                ax.set_title(f"L[{i},{j}]", fontsize=7)
                latent_idx += 1
            else:
                ax.axis("off")
            ax.grid(True)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=n_chains,
               bbox_to_anchor=(0.5, 0.02))
    plt.suptitle(f"MCMC Trace: {param_name} — Component {k_idx}", fontsize=16)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.show()


for k in range(K):
    plot_cholesky_traces(posterior_samples_sorted, n_params=P, k_idx=k)

In [ ]:
# ── Cell 7 : Posterior Summary ────────────────────────────────────────────────

# ── Mixture weights ────────────────────────────────────────────────────────────
pvec_flat = posterior_samples_sorted["pvec"].reshape(-1, K)

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(pvec_flat, labels=[f"Comp {k+1}" for k in range(K)],
           patch_artist=True,
           boxprops=dict(facecolor="#4682B4", alpha=0.6))
ax.set_ylabel("Mixture weight")
ax.set_title("Posterior Distribution of Mixture Weights (pvec)")
plt.tight_layout()
plt.show()

print("\n=== Posterior Mean Mixture Weights ===")
print(np.round(pvec_flat.mean(axis=0), 4))

# ── Component means (mu_k) ─────────────────────────────────────────────────────
mu_flat = posterior_samples_sorted["mu_k"].reshape(-1, K, P)

print("\n=== Posterior Mean mu_k ===")
display(pd.DataFrame(
    mu_flat.mean(axis=0),
    index=[f"Comp_{k+1}" for k in range(K)],
    columns=param_names,
).round(4))

fig, axes = plt.subplots(K, P, figsize=(3 * P, 3 * K), sharey=False)
colors = sns.color_palette("tab10", K)

for k in range(K):
    for p in range(P):
        ax      = axes[k, p]
        samples = mu_flat[:, k, p]
        sns.kdeplot(samples, ax=ax, fill=True, color=colors[k], alpha=0.4, lw=2)
        ax.axvline(np.mean(samples), color=colors[k], lw=1.5)
        ax.axvline(0, color="black", lw=0.8, linestyle="--", alpha=0.4)
        ax.set_yticks([])
        sns.despine(ax=ax, left=True)
        if k == 0:
            ax.set_title(param_names[p], fontsize=10, fontweight="bold")
        if p == 0:
            ax.set_ylabel(f"Comp {k+1}", fontsize=10, fontweight="bold")

plt.suptitle("Posterior Marginal Densities: mu_k per Component", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# ── Delta ──────────────────────────────────────────────────────────────────────
delta_flat = posterior_samples["Delta"].reshape(-1, D, P)

print("\n=== Posterior Mean Delta ===")
display(pd.DataFrame(
    delta_flat.mean(axis=0),
    index=demo_names,
    columns=param_names,
).round(4))

fig, axes = plt.subplots(D, P, figsize=(3 * P, 3 * D), sharey=False)
if D == 1:
    axes = axes[np.newaxis, :]

for d in range(D):
    for p in range(P):
        ax      = axes[d, p]
        samples = delta_flat[:, d, p]
        sns.kdeplot(samples, ax=ax, fill=True, color="#5B5EA6", alpha=0.4, lw=2)
        ax.axvline(np.mean(samples), color="#5B5EA6", lw=1.5)
        ax.axvline(0, color="black", lw=0.8, linestyle="--", alpha=0.4)
        ax.set_yticks([])
        sns.despine(ax=ax, left=True)
        if d == 0:
            ax.set_title(param_names[p], fontsize=10, fontweight="bold")
        if p == 0:
            ax.set_ylabel(demo_names[d], fontsize=10, fontweight="bold")

plt.suptitle("Posterior Marginal Densities: Delta (Demographics)", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8 : Posterior Summary ────────────────────────────────────────────────

# ── Mixture weights ────────────────────────────────────────────────────────────
pvec_flat = posterior_samples_sorted["pvec"].reshape(-1, K)

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(pvec_flat, labels=[f"Comp {k+1}" for k in range(K)],
           patch_artist=True,
           boxprops=dict(facecolor="#4682B4", alpha=0.6))
ax.set_ylabel("Mixture weight")
ax.set_title("Posterior Distribution of Mixture Weights (pvec)")
plt.tight_layout()
plt.show()

print("\n=== Posterior Mean Mixture Weights ===")
print(np.round(pvec_flat.mean(axis=0), 4))

# ── Component means (mu_k) ─────────────────────────────────────────────────────
mu_flat = posterior_samples_sorted["mu_k"].reshape(-1, K, P)

print("\n=== Posterior Mean mu_k ===")
mu_mean_df = pd.DataFrame(
    mu_flat.mean(axis=0),
    index=[f"Comp_{k+1}" for k in range(K)],
    columns=param_names
)
display(mu_mean_df.round(4))

# ── Marginal density grid for mu_k ────────────────────────────────────────────
fig, axes = plt.subplots(K, P, figsize=(3 * P, 3 * K), sharey=False)
colors = sns.color_palette("tab10", K)

for k in range(K):
    for p in range(P):
        ax      = axes[k, p]
        samples = mu_flat[:, k, p]
        sns.kdeplot(samples, ax=ax, fill=True, color=colors[k], alpha=0.4, lw=2)
        ax.axvline(np.mean(samples), color=colors[k], lw=1.5)
        ax.axvline(0,                color="black",   lw=0.8, linestyle="--", alpha=0.4)
        ax.set_yticks([])
        sns.despine(ax=ax, left=True)
        if k == 0:
            ax.set_title(param_names[p], fontsize=10, fontweight="bold")
        if p == 0:
            ax.set_ylabel(f"Comp {k+1}", fontsize=10, fontweight="bold")

plt.suptitle("Posterior Marginal Densities: mu_k per Component", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# ── Delta recovery ─────────────────────────────────────────────────────────────
delta_flat = posterior_samples["Delta"].reshape(-1, D, P)

print("\n=== Posterior Mean Delta (demographics → params) ===")
delta_mean_df = pd.DataFrame(
    delta_flat.mean(axis=0),
    index=demo_names,
    columns=param_names
)
display(delta_mean_df.round(4))

fig, axes = plt.subplots(D, P, figsize=(3 * P, 3 * D), sharey=False)
if D == 1:
    axes = axes[np.newaxis, :]

for d in range(D):
    for p in range(P):
        ax      = axes[d, p]
        samples = delta_flat[:, d, p]
        sns.kdeplot(samples, ax=ax, fill=True, color="#5B5EA6", alpha=0.4, lw=2)
        ax.axvline(np.mean(samples), color="#5B5EA6", lw=1.5)
        ax.axvline(0, color="black", lw=0.8, linestyle="--", alpha=0.4)
        ax.set_yticks([])
        sns.despine(ax=ax, left=True)
        if d == 0:
            ax.set_title(param_names[p], fontsize=10, fontweight="bold")
        if p == 0:
            ax.set_ylabel(demo_names[d], fontsize=10, fontweight="bold")

plt.suptitle("Posterior Marginal Densities: Delta (Demographics)", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()